# Fall 2025 DS701 Grade Calculation



Final grades will be computed based on the following:<br>

| Percentage | Category |
| ---------- | -------- |
| 15%        | Participation and in-class activities |
| 15%        | Homework assignments |
| 30%        | Midterm challenge |
| 40%        | Final Project (where 24% is for the project and 16% is for the personal contributions)  |


In [15]:
import pandas as pd

df = pd.read_csv('DS701_Fall_2025_grades.csv')


In [16]:
print(df.columns)


Index(['Name', 'SID', 'Email', 'Sections', 'Linear Algebra',
       'Linear Algebra - Max Points', 'Linear Algebra - Submission Time',
       'Linear Algebra - Lateness (H:M:S)', 'Pandas and Scikit-Learn',
       'Pandas and Scikit-Learn - Max Points',
       'Pandas and Scikit-Learn - Submission Time',
       'Pandas and Scikit-Learn - Lateness (H:M:S)',
       'Project Final Submission, Repo, Presentation and Report',
       'Project Final Submission, Repo, Presentation and Report - Max Points',
       'Project Final Submission, Repo, Presentation and Report - Submission Time',
       'Project Final Submission, Repo, Presentation and Report - Lateness (H:M:S)',
       'Probability, Statistics and Distance',
       'Probability, Statistics and Distance - Max Points',
       'Probability, Statistics and Distance - Submission Time',
       'Probability, Statistics and Distance - Lateness (H:M:S)',
       'K-Means Clustering', 'K-Means Clustering - Max Points',
       'K-Means Clustering

## Lateness and Penalty Calculations

In [17]:
import re

def parse_lateness_to_hours(lateness_str):
    """Convert H:M:S lateness string to total hours as float."""
    if pd.isna(lateness_str) or lateness_str == '00:00:00':
        return 0.0
    # Parse H:M:S format
    parts = str(lateness_str).split(':')
    if len(parts) == 3:
        hours = int(parts[0])
        minutes = int(parts[1])
        seconds = int(parts[2])
        return hours + minutes / 60 + seconds / 3600
    return 0.0

def apply_late_penalty(score, lateness_hours):
    """Apply 10% penalty if lateness is > 0 and < 48 hours."""
    if pd.isna(score):
        return 0.0
    if lateness_hours > 0 and lateness_hours < 48:
        return score * 0.9
    return score


## Find and Assign Categories to Assignments

In [18]:
# Get all unique assignment names by finding columns that don't have suffixes
assignment_columns = []
for col in df.columns:
    if ' - Max Points' in col:
        assignment_name = col.replace(' - Max Points', '')
        assignment_columns.append(assignment_name)

print("Found assignments:")
for a in assignment_columns:
    print(f"  - {a}")


Found assignments:
  - Linear Algebra
  - Pandas and Scikit-Learn
  - Project Final Submission, Repo, Presentation and Report
  - Probability, Statistics and Distance
  - K-Means Clustering
  - Hierarchical Clustering and GMMs
  - Spark Midterm Survey
  - AI Use Reflection
  - Intro Survey
  - Spark Project Pitches and Preference Form
  - Team Agreement
  - Project Definition, Research and Problem Understanding
  - Exploratory Data Analysis
  - Project Proof of Concept
  - Pandas in-class activity
  - Exploratory Data Analysis -- Individual Reflection
  - Midterm Challenge -- Sports
  - Midterm Challenge -- Health
  - PoC -- Individual Reflection
  - Final Project Personal Contributions
  - Project Definition Personal Reflections


In [19]:
# Define assignment categories
midterm_challenge_assignments = [
    'Midterm Challenge -- Sports',
    'Midterm Challenge -- Health'
]

final_project_assignments = [
    'Project Final Submission, Repo, Presentation and Report',
    'Final Project Personal Contributions'
]

# In-class activities = specific assignments plus any with 'in-class activity' in name
in_class_specific = [
    'AI Use Reflection',
    'Intro Survey',
    'Spark Project Pitches and Preference Form',
    'Spark Midterm Survey'
]

in_class_activities = [
    a for a in assignment_columns 
    if 'in-class activity' in a.lower() or a in in_class_specific
]

# Homework = everything except midterm challenge, final project, and in-class activities
homework_assignments = [
    a for a in assignment_columns 
    if a not in midterm_challenge_assignments 
    and a not in final_project_assignments
    and a not in in_class_activities
]

print("Homework Assignments:")
for a in homework_assignments:
    print(f"  - {a}")
    
print("\nIn-Class Activities:")
for a in in_class_activities:
    print(f"  - {a}")
    
print("\nMidterm Challenge Assignments:")
for a in midterm_challenge_assignments:
    print(f"  - {a}")
    
print("\nFinal Project Assignments:")
for a in final_project_assignments:
    print(f"  - {a}")


Homework Assignments:
  - Linear Algebra
  - Pandas and Scikit-Learn
  - Probability, Statistics and Distance
  - K-Means Clustering
  - Hierarchical Clustering and GMMs
  - Team Agreement
  - Project Definition, Research and Problem Understanding
  - Exploratory Data Analysis
  - Project Proof of Concept
  - Exploratory Data Analysis -- Individual Reflection
  - PoC -- Individual Reflection
  - Project Definition Personal Reflections

In-Class Activities:
  - Spark Midterm Survey
  - AI Use Reflection
  - Intro Survey
  - Spark Project Pitches and Preference Form
  - Pandas in-class activity

Midterm Challenge Assignments:
  - Midterm Challenge -- Sports
  - Midterm Challenge -- Health

Final Project Assignments:
  - Project Final Submission, Repo, Presentation and Report
  - Final Project Personal Contributions


In [20]:
# Calculate scores with late penalties applied
def get_adjusted_score(row, assignment_name):
    """Get the score for an assignment with late penalty applied if applicable."""
    score_col = assignment_name
    lateness_col = f"{assignment_name} - Lateness (H:M:S)"
    
    if score_col not in df.columns:
        return 0.0
    
    score = row[score_col]
    if pd.isna(score):
        return 0.0
    
    # Ignore lateness for these assignment types
    if 'Individual Reflection' in assignment_name:
        return score
    if 'Personal Contributions' in assignment_name:
        return score
    if 'Project Final Submission' in assignment_name:
        return score
    
    # Get lateness if available
    if lateness_col in df.columns:
        lateness_hours = parse_lateness_to_hours(row[lateness_col])
        return apply_late_penalty(score, lateness_hours)
    else:
        return score

# Calculate Total Homework Score (with late penalties)
def calculate_homework_total(row):
    total = 0.0
    for assignment in homework_assignments:
        total += get_adjusted_score(row, assignment)
    return total

df['Total Homework Score'] = df.apply(calculate_homework_total, axis=1)

# Calculate In-Class Activities Total (with penalties)
def calculate_in_class_total(row):
    total = 0.0
    for assignment in in_class_activities:
        total += get_adjusted_score(row, assignment)
    return total

df['In-Class Activities Score'] = df.apply(calculate_in_class_total, axis=1)

# Calculate Final Project Total (sum of Final Project + Personal Contributions, with penalties)
def calculate_final_project_total(row):
    final_project_score = get_adjusted_score(row, 'Project Final Submission, Repo, Presentation and Report')
    personal_contributions_score = get_adjusted_score(row, 'Final Project Personal Contributions')
    return final_project_score + personal_contributions_score

df['Final Project Total Score'] = df.apply(calculate_final_project_total, axis=1)

# Calculate max possible points for each category
max_homework_points = 0
for assignment in homework_assignments:
    max_col = f"{assignment} - Max Points"
    if max_col in df.columns:
        max_val = df[max_col].max()
        if pd.notna(max_val):
            max_homework_points += max_val

max_in_class_points = 0
for assignment in in_class_activities:
    max_col = f"{assignment} - Max Points"
    if max_col in df.columns:
        max_val = df[max_col].max()
        if pd.notna(max_val):
            max_in_class_points += max_val

max_sports_points = df['Midterm Challenge -- Sports - Max Points'].max() if 'Midterm Challenge -- Sports - Max Points' in df.columns else 0
max_health_points = df['Midterm Challenge -- Health - Max Points'].max() if 'Midterm Challenge -- Health - Max Points' in df.columns else 0

max_final_project_points = 0
if 'Project Final Submission, Repo, Presentation and Report - Max Points' in df.columns:
    max_final_project_points += df['Project Final Submission, Repo, Presentation and Report - Max Points'].max()
if 'Final Project Personal Contributions - Max Points' in df.columns:
    max_final_project_points += df['Final Project Personal Contributions - Max Points'].max()

# Calculate percentages for each category
df['Total Homework Percentage'] = (df['Total Homework Score'] / max_homework_points * 100) if max_homework_points > 0 else 0
df['In-Class Activities Percentage'] = (df['In-Class Activities Score'] / max_in_class_points * 100) if max_in_class_points > 0 else 0
df['Final Project Percentage'] = (df['Final Project Total Score'] / max_final_project_points * 100) if max_final_project_points > 0 else 0

# Calculate Personal Contributions score and percentage (out of 40 points)
def get_personal_contributions_score(row):
    return get_adjusted_score(row, 'Final Project Personal Contributions')

df['Final Project Personal Contributions Score'] = df.apply(get_personal_contributions_score, axis=1)
max_personal_contributions = 40.0
df['Final Project Personal Contributions Percentage'] = (df['Final Project Personal Contributions Score'] / max_personal_contributions * 100)

# Calculate Midterm Challenge Score (highest percentage of Sports or Health, with penalties)
def calculate_midterm_challenge(row):
    sports_score = get_adjusted_score(row, 'Midterm Challenge -- Sports')
    health_score = get_adjusted_score(row, 'Midterm Challenge -- Health')
    
    # Calculate percentages
    sports_pct = (sports_score / max_sports_points * 100) if max_sports_points > 0 and sports_score > 0 else 0
    health_pct = (health_score / max_health_points * 100) if max_health_points > 0 and health_score > 0 else 0
    
    # Return the score with the highest percentage
    if sports_pct >= health_pct:
        return sports_score, sports_pct, max_sports_points
    else:
        return health_score, health_pct, max_health_points

# Apply the function and unpack results
midterm_results = df.apply(calculate_midterm_challenge, axis=1, result_type='expand')
df['Midterm Challenge Score'] = midterm_results[0]
df['Midterm Challenge Percentage'] = midterm_results[1]
df['Midterm Challenge Max Points'] = midterm_results[2]

print("New columns added:")
print("  - Total Homework Score")
print("  - Total Homework Percentage")
print("  - In-Class Activities Score")
print("  - In-Class Activities Percentage")
print("  - Midterm Challenge Score (selected by highest percentage)")
print("  - Midterm Challenge Percentage")
print("  - Midterm Challenge Max Points (for selected option)")
print("  - Final Project Total Score")
print("  - Final Project Percentage")
print("  - Final Project Personal Contributions Score")
print("  - Final Project Personal Contributions Percentage (out of 40)")


New columns added:
  - Total Homework Score
  - Total Homework Percentage
  - In-Class Activities Score
  - In-Class Activities Percentage
  - Midterm Challenge Score (selected by highest percentage)
  - Midterm Challenge Percentage
  - Midterm Challenge Max Points (for selected option)
  - Final Project Total Score
  - Final Project Percentage
  - Final Project Personal Contributions Score
  - Final Project Personal Contributions Percentage (out of 40)


In [21]:
# Debug: Check max points calculation for homework
print("=== HOMEWORK MAX POINTS BREAKDOWN ===\n")
print(f"Total max homework points used: {max_homework_points}\n")

print("Individual homework assignments and their max points:")
homework_max_breakdown = []
for assignment in homework_assignments:
    max_col = f"{assignment} - Max Points"
    if max_col in df.columns:
        max_val = df[max_col].max()
        if pd.notna(max_val):
            homework_max_breakdown.append((assignment, max_val))
            print(f"  {assignment}: {max_val}")
        else:
            print(f"  {assignment}: NO MAX POINTS DATA")
    else:
        print(f"  {assignment}: MAX POINTS COLUMN NOT FOUND")

print(f"\nSum of all homework max points: {sum([x[1] for x in homework_max_breakdown])}")

# Check for students with >100% homework
over_100 = df[df['Total Homework Percentage'] > 100][['Name', 'Total Homework Score', 'Total Homework Percentage']].head(5)
if len(over_100) > 0:
    print(f"\n=== STUDENTS WITH >100% HOMEWORK ===")
    print(over_100)
else:
    print("\n=== No students with >100% homework ===")

# Show a sample student's homework breakdown
print("\n=== SAMPLE STUDENT HOMEWORK BREAKDOWN ===")
sample_student = df.iloc[0]
print(f"Student: {sample_student['Name']}")
print(f"Total Homework Score: {sample_student['Total Homework Score']}")
print(f"Total Homework Percentage: {sample_student['Total Homework Percentage']:.2f}%")
print("\nIndividual homework assignments:")
for assignment in homework_assignments[:5]:  # Show first 5
    score = sample_student[assignment] if assignment in df.columns else 'N/A'
    max_col = f"{assignment} - Max Points"
    max_pts = sample_student[max_col] if max_col in df.columns else 'N/A'
    print(f"  {assignment}: {score}/{max_pts}")


=== HOMEWORK MAX POINTS BREAKDOWN ===

Total max homework points used: 239.0

Individual homework assignments and their max points:
  Linear Algebra: 19.0
  Pandas and Scikit-Learn: 53.0
  Probability, Statistics and Distance: 8.0
  K-Means Clustering: 21.0
  Hierarchical Clustering and GMMs: 45.0
  Team Agreement: 10.0
  Project Definition, Research and Problem Understanding: 20.0
  Exploratory Data Analysis: 20.0
  Project Proof of Concept: 25.0
  Exploratory Data Analysis -- Individual Reflection: 5.0
  PoC -- Individual Reflection: 5.0
  Project Definition Personal Reflections: 8.0

Sum of all homework max points: 239.0

=== No students with >100% homework ===

=== SAMPLE STUDENT HOMEWORK BREAKDOWN ===
Student: Aadi Sharma
Total Homework Score: 238.0
Total Homework Percentage: 99.58%

Individual homework assignments:
  Linear Algebra: 19.0/19.0
  Pandas and Scikit-Learn: 53.0/53.0
  Probability, Statistics and Distance: 8.0/8.0
  K-Means Clustering: 20.0/21.0
  Hierarchical Cluster

In [22]:
# Check which assignments have scores exceeding max points (extra credit)
print("=== CHECKING FOR EXTRA CREDIT ===\n")

for assignment in homework_assignments:
    max_col = f"{assignment} - Max Points"
    if assignment in df.columns and max_col in df.columns:
        # Find students who scored above max
        max_points = df[max_col].max()
        if pd.notna(max_points):
            above_max = df[df[assignment] > max_points]
            if len(above_max) > 0:
                print(f"{assignment}:")
                print(f"  Max points: {max_points}")
                print(f"  Students scoring above max: {len(above_max)}")
                print(f"  Highest score: {df[assignment].max()}")
                print()

# Show Aadi Sharma's full homework breakdown
print("\n=== AADI SHARMA'S COMPLETE HOMEWORK BREAKDOWN ===")
aadi = df[df['Name'] == 'Aadi Sharma'].iloc[0]
total_calculated = 0
for assignment in homework_assignments:
    if assignment in df.columns:
        score = aadi[assignment] if pd.notna(aadi[assignment]) else 0
        max_col = f"{assignment} - Max Points"
        max_pts = aadi[max_col] if max_col in df.columns and pd.notna(aadi[max_col]) else 0
        total_calculated += score
        extra = " (EXTRA CREDIT!)" if score > max_pts else ""
        print(f"  {assignment}: {score}/{max_pts}{extra}")

print(f"\nTotal calculated: {total_calculated}")
print(f"Total in dataframe: {aadi['Total Homework Score']}")
print(f"Max possible (no extra credit): {max_homework_points}")


=== CHECKING FOR EXTRA CREDIT ===


=== AADI SHARMA'S COMPLETE HOMEWORK BREAKDOWN ===
  Linear Algebra: 19.0/19.0
  Pandas and Scikit-Learn: 53.0/53.0
  Probability, Statistics and Distance: 8.0/8.0
  K-Means Clustering: 20.0/21.0
  Hierarchical Clustering and GMMs: 45.0/45.0
  Team Agreement: 10.0/10.0
  Project Definition, Research and Problem Understanding: 20.0/20.0
  Exploratory Data Analysis: 20.0/20.0
  Project Proof of Concept: 25.0/25.0
  Exploratory Data Analysis -- Individual Reflection: 5.0/5.0
  PoC -- Individual Reflection: 5.0/5.0
  Project Definition Personal Reflections: 8.0/8.0

Total calculated: 238.0
Total in dataframe: 238.0
Max possible (no extra credit): 239.0


In [23]:
# Display the new columns with student info
result_df = df[['Name', 'SID', 'Email', 
                'Total Homework Score', 'Total Homework Percentage',
                'In-Class Activities Score', 'In-Class Activities Percentage', 
                'Midterm Challenge Score', 'Midterm Challenge Percentage', 'Midterm Challenge Max Points',
                'Final Project Total Score', 'Final Project Percentage',
                'Final Project Personal Contributions Score', 'Final Project Personal Contributions Percentage']]

print(f"Total students: {len(result_df)}")
print(f"\nHomework assignments count: {len(homework_assignments)}")
print(f"In-class activities count: {len(in_class_activities)}")
            
print(f"\nMax possible homework points: {max_homework_points}")
print(f"Max possible in-class activities points: {max_in_class_points}")
print(f"Max Midterm Challenge points: Sports={max_sports_points}, Health={max_health_points}")
print(f"Max Final Project points: {max_final_project_points}")
print(f"Max Personal Contributions points: {max_personal_contributions}")

result_df.head(20)


Total students: 108

Homework assignments count: 12
In-class activities count: 5

Max possible homework points: 239.0
Max possible in-class activities points: 7.0
Max Midterm Challenge points: Sports=210.0, Health=220.0
Max Final Project points: 100.0
Max Personal Contributions points: 40.0


,Name,SID,Email,Total Homework Score,Total Homework Percentage,In-Class Activities Score,In-Class Activities Percentage,Midterm Challenge Score,Midterm Challenge Percentage,Midterm Challenge Max Points,Final Project Total Score,Final Project Percentage,Final Project Personal Contributions Score,Final Project Personal Contributions Percentage
0,Aadi Sharma,U16580073,aadigs@bu.edu,238.0,99.581590,6.0,85.714286,214.0,101.904762,210.0,97.0,97.0,39.0,97.5
1,Aastha Kishorebhai Gidwani,U22349929,aastha36@bu.edu,237.0,99.163180,7.0,100.000000,205.0,97.619048,210.0,100.0,100.0,40.0,100.0
2,Abdul Majid,U40945413,abdulmds@bu.edu,238.0,99.581590,6.0,85.714286,230.0,104.545455,220.0,96.0,96.0,36.0,90.0
3,Abhishek Sharma,U57955711,abhibgru@bu.edu,236.0,98.744770,7.0,100.000000,203.0,96.666667,210.0,94.0,94.0,34.0,85.0
4,Achuthan Rathinam,U40915317,achuthan@bu.edu,217.0,90.794979,6.0,85.714286,240.0,109.090909,220.0,100.0,100.0,40.0,100.0
5,Aidan Smith,U46699295,aidansmi@bu.edu,236.0,98.744770,7.0,100.000000,213.0,101.428571,210.0,100.0,100.0,40.0,100.0
6,Aijia Zhang,U69117321,ajzh0517@bu.edu,239.0,100.000000,7.0,100.000000,209.0,99.523810,210.0,100.0,100.0,40.0,100.0
7,Akhil Elamanchili,U93690454,aelamanc@bu.edu,231.0,96.652720,7.0,100.000000,208.0,99.047619,210.0,100.0,100.0,40.0,100.0
8,Albert Dahlberg,U91649759,aericd@bu.edu,225.5,94.351464,7.0,100.000000,227.0,103.181818,220.0,100.0,100.0,40.0,100.0
9,Albert Hoe,U41215502,alhoe@bu.edu,217.0,90.794979,7.0,100.000000,209.0,99.523810,210.0,68.0,68.0,10.0,25.0


In [24]:
# Show examples of late penalty applications
# Find students who had late submissions within the 48-hour penalty window

print("Examples of late penalty applications (lateness > 0 and < 48 hours):")
print("(Note: 'Individual Reflection', 'Personal Contributions', and 'Project Final Submission' assignments are excluded from late penalties)\n")

# Check a few specific assignments for late penalties
for assignment in ['Linear Algebra', 'K-Means Clustering', 'Hierarchical Clustering and GMMs']:
    score_col = assignment
    lateness_col = f"{assignment} - Lateness (H:M:S)"
    
    if lateness_col in df.columns:
        # Find students with late submissions in penalty range
        for idx, row in df.iterrows():
            lateness_hours = parse_lateness_to_hours(row[lateness_col])
            if lateness_hours > 0 and lateness_hours < 48:
                original_score = row[score_col]
                adjusted_score = apply_late_penalty(original_score, lateness_hours)
                print(f"{row['Name']} - {assignment}:")
                print(f"  Original score: {original_score}, Lateness: {row[lateness_col]} ({lateness_hours:.2f}h)")
                print(f"  Adjusted score: {adjusted_score} (10% penalty applied)")
                print()
                break  # Just show one example per assignment


Examples of late penalty applications (lateness > 0 and < 48 hours):
(Note: 'Individual Reflection', 'Personal Contributions', and 'Project Final Submission' assignments are excluded from late penalties)

Shiwen Zhang - Linear Algebra:
  Original score: 19.0, Lateness: 18:21:08 (18.35h)
  Adjusted score: 17.1 (10% penalty applied)

Dong Yimeng - K-Means Clustering:
  Original score: 20.0, Lateness: 41:07:32 (41.13h)
  Adjusted score: 18.0 (10% penalty applied)

Albert Dahlberg - Hierarchical Clustering and GMMs:
  Original score: 45.0, Lateness: 15:48:50 (15.81h)
  Adjusted score: 40.5 (10% penalty applied)



In [25]:
# Summary statistics for the new score columns
print("Summary Statistics for Scores:\n")
score_cols = ['Total Homework Score', 'In-Class Activities Score', 'Midterm Challenge Score', 'Final Project Total Score', 'Final Project Personal Contributions Score']
print(df[score_cols].describe())

print("\n\nSummary Statistics for Percentages:\n")
pct_cols = ['Total Homework Percentage', 'In-Class Activities Percentage', 'Midterm Challenge Percentage', 'Final Project Percentage', 'Final Project Personal Contributions Percentage']
print(df[pct_cols].describe())


Summary Statistics for Scores:

       Total Homework Score  In-Class Activities Score  \
count            108.000000                 108.000000   
mean             230.885185                   6.394444   
std               21.358191                   1.159748   
min               38.000000                   1.000000   
25%              232.500000                   6.000000   
50%              236.000000                   7.000000   
75%              238.000000                   7.000000   
max              239.000000                   7.000000   

       Midterm Challenge Score  Final Project Total Score  \
count               108.000000                 108.000000   
mean                216.111111                  97.046296   
std                  15.235051                  10.579812   
min                 164.000000                   0.000000   
25%                 205.000000                  98.000000   
50%                 215.500000                  99.000000   
75%               

In [26]:
# Find students with late submissions past 48 hours
# (excluding certain assignment types that don't have late penalties)
print("Students with late submissions past 48 hours:\n")
print("(Excluding 'Individual Reflection', 'Personal Contributions', and 'Project Final Submission' assignments)\n")

late_records = []

for assignment in assignment_columns:
    # Skip assignments that don't have late penalties
    if 'Individual Reflection' in assignment:
        continue
    if 'Personal Contributions' in assignment:
        continue
    if 'Personal Reflections' in assignment:
        continue
    if 'Project Final Submission' in assignment:
        continue
    
    lateness_col = f"{assignment} - Lateness (H:M:S)"
    score_col = assignment
    
    if lateness_col in df.columns:
        for idx, row in df.iterrows():
            lateness_hours = parse_lateness_to_hours(row[lateness_col])
            if lateness_hours >= 48:
                late_records.append({
                    'Name': row['Name'],
                    'Email': row['Email'],
                    'Assignment': assignment,
                    'Score': row[score_col],
                    'Max Points': row[f"{assignment} - Max Points"] if f"{assignment} - Max Points" in df.columns else None,
                    'Lateness': row[lateness_col],
                    'Lateness (hours)': round(lateness_hours, 2)
                })

# Create a dataframe of late submissions
late_df = pd.DataFrame(late_records)

if len(late_df) > 0:
    # Sort by student name then assignment
    late_df = late_df.sort_values(['Name', 'Assignment'])
    print(f"Total late submissions (>= 48 hours): {len(late_df)}")
    print(f"Number of unique students affected: {late_df['Name'].nunique()}")
    print()
    
    # Display all late submissions
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_colwidth', 50)
    display(late_df)
else:
    print("No late submissions past 48 hours found.")


Students with late submissions past 48 hours:

(Excluding 'Individual Reflection', 'Personal Contributions', and 'Project Final Submission' assignments)

Total late submissions (>= 48 hours): 9
Number of unique students affected: 8



,Name,Email,Assignment,Score,Max Points,Lateness,Lateness (hours)
0,Albert Hoe,alhoe@bu.edu,Pandas and Scikit-Learn,38.0,53.0,335:38:22,335.64
3,Ching Hsuan Lin,shawn17@bu.edu,Hierarchical Clustering and GMMs,45.0,45.0,84:03:54,84.06
4,Khushi Anil Teli,tkhushi@bu.edu,Hierarchical Clustering and GMMs,45.0,45.0,379:41:21,379.69
2,Pin-Hao Pan,pp0126@bu.edu,K-Means Clustering,20.0,21.0,757:36:23,757.61
5,San Yan,sanyan@bu.edu,Hierarchical Clustering and GMMs,45.0,45.0,354:13:47,354.23
1,San Yan,sanyan@bu.edu,Pandas and Scikit-Learn,53.0,53.0,253:33:11,253.55
6,Sodbileg Ganbat,sodbileg@bu.edu,Hierarchical Clustering and GMMs,10.0,45.0,373:22:14,373.37
7,Yang Lu,ylu7@bu.edu,AI Use Reflection,3.0,3.0,172:27:18,172.45
8,Yuki Li,yukili@bu.edu,Intro Survey,1.0,1.0,136:03:12,136.05


In [27]:
# Write the entire dataframe to an Excel file
output_file = 'DS701_Fall_2025_grades_processed.xlsx'

df.to_excel(output_file, index=False, engine='openpyxl')

print(f"Dataframe written to: {output_file}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")



Dataframe written to: DS701_Fall_2025_grades_processed.xlsx
Total rows: 108
Total columns: 100


In [28]:
# Write summary scores to a separate Excel file
summary_columns = [
    'Name', 'SID', 'Email', 'Sections',
    'Total Homework Score', 'Total Homework Percentage',
    'In-Class Activities Score', 'In-Class Activities Percentage',
    'Midterm Challenge Score', 'Midterm Challenge Percentage', 'Midterm Challenge Max Points',
    'Final Project Total Score', 'Final Project Percentage',
    'Final Project Personal Contributions Score', 'Final Project Personal Contributions Percentage'
]

# Select only columns that exist in the dataframe
available_summary_cols = [col for col in summary_columns if col in df.columns]

summary_df = df[available_summary_cols]
summary_output_file = 'summary_scores.xlsx'

summary_df.to_excel(summary_output_file, index=False, engine='openpyxl')

print(f"\nSummary scores written to: {summary_output_file}")
print(f"Columns included: {len(available_summary_cols)}")
print(f"  - {', '.join(available_summary_cols[:4])} (identifiers)")
print(f"  - {len(available_summary_cols) - 4} score/percentage columns")
print(f"\nIncludes Personal Contributions: Score (out of 40) and Percentage")



Summary scores written to: summary_scores.xlsx
Columns included: 15
  - Name, SID, Email, Sections (identifiers)
  - 11 score/percentage columns

Includes Personal Contributions: Score (out of 40) and Percentage
